<a href="https://colab.research.google.com/github/pressleydavid/hw5/blob/main/HW6_ST554_DavidPressley.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW6
**David Pressley**

**Course:** ST554

**Due:** March 3, 2026  

**Submission:** Save to GitHub repo, submit repo link on Moodle

The purpose of this homework is to get a little more practice with SQL and gain some practice with creating classes.

**Instructions:**
- Include markdown text describing what you are doing, even when not explicitly asked for
- Add comments to .py file explaining work
- Save updates to GitHub as you work through it
- No edits after the due date
- check sharing settings
- use hw5 repo. Develop on main.

In [22]:
# import ability to use Markdown in Code Cell displays
from IPython.display import display, Markdown

## Part I - More Practice Querying a Database (16 pts)
The purpose of this homework is to get a little more practice with SQL and gain some practice with creating classes.

### Question 1
Connect to the database and then look at all of the tables in the database (use `read_sql()` from `pandas` to have this returned as a data frame). (2 pts)

In [23]:
import pandas as pd
import sqlite3
import urllib.request

url = "https://github.com/jknecht/baseball-archive-sqlite/releases/download/2022/lahman_1871-2022.sqlite"
output_file = "lahman.sqlite"

urllib.request.urlretrieve(url, output_file)

con = sqlite3.connect('lahman.sqlite')
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", con)
display(Markdown(f"### Tables in the Lahman Database"))
print(tables)


### Tables in the Lahman Database

                   name
0           AllstarFull
1           Appearances
2        AwardsManagers
3         AwardsPlayers
4   AwardsShareManagers
5    AwardsSharePlayers
6               Batting
7           BattingPost
8        CollegePlaying
9              Fielding
10           FieldingOF
11      FieldingOFsplit
12         FieldingPost
13           HallOfFame
14            HomeGames
15             Managers
16         ManagersHalf
17                Parks
18               People
19             Pitching
20         PitchingPost
21             Salaries
22              Schools
23           SeriesPost
24                Teams
25      TeamsFranchises
26            TeamsHalf


### Question 2
Using SQL, construct a table of hall of fame pitchers (any hall of famer that pitched) that gives the `playerID` and their total (sum) for `GS`, `G`, `W`, `L`, `IPOuts`, `CG`, `SHO`, and `SV` columns. The summing can be done in `pandas` or in the SQL call. (6 pts)

In [24]:
# LHS is HallofFame where inducted
# RHS is Pitchers
# inner join (default join) to get pitchers inducted
hof_pitchers = pd.read_sql("""
    SELECT
        l.playerID,
        l.inducted,
        SUM(r.GS) as total_games_started,
        SUM(r.G) as total_games,
        SUM(r.W) as total_wins,
        SUM(r.L) as total_losses,
        SUM(r.IPOuts) as total_outs_pitched,
        SUM(r.CG) as total_complete_games,
        SUM(r.SHO) as total_shutouts,
        SUM(r.SV) as total_saves
    FROM (
        SELECT DISTINCT playerID, inducted
        FROM HallofFame
        WHERE inducted = 'Y'
    ) as l
    INNER JOIN
    Pitching as r
    ON l.playerID = r.playerID
    GROUP BY l.playerID;
    """, con)
display(Markdown(f"### Selected totals for pitchers inducted into Hall of Fame"))
display(hof_pitchers)

### Selected totals for pitchers inducted into Hall of Fame

,playerID,inducted,total_games_started,total_games,total_wins,total_losses,total_outs_pitched,total_complete_games,total_shutouts,total_saves
0,alexape01,Y,599,696,373,208,15570,437,90,32
1,ansonca01,Y,0,3,0,1,12,0,0,1
2,becklja01,Y,1,1,0,1,12,0,0,0
3,bendech01,Y,334,459,212,127,9051,255,40,34
4,blylebe01,Y,685,692,287,250,14910,242,60,0
...,...,...,...,...,...,...,...,...,...,...
103,willivi01,Y,471,513,249,205,11988,388,50,11
104,wrighge01,Y,0,3,0,1,15,0,0,0
105,wrighha01,Y,8,36,4,4,301,0,0,14
106,wynnea01,Y,612,691,300,244,13692,290,49,15


### Question 3
For all of the hall of fame pitchers, use SQL to create a table of their batting statistics. Namely, the `playerID` and their total (sum) for `AB`, `R`, `H`, `HR`, `RBI`, `BB`, and `SO`. The summing can be done in `pandas` or in the SQL call. (4 pts)

In [25]:
# to get all hall of fame pitchers from sql query, we need to select records from a subsetting query for distinct players inducted as the left-hand table(l).
# then join that table with the "middle table" (m) as a junction table to bridge between HoF and Batting stats, linking them together with playerID
# finally, return the totals from the right-hand table (r)
hof_pitch_bat_stat = pd.read_sql("""
    SELECT
        l.playerID,
        l.inducted,
        SUM(r.AB) as total_at_bats,
        SUM(r.R) as total_runs,
        SUM(r.H) as total_hits,
        SUM(r.HR) as total_home_runs,
        SUM(r.RBI) as total_RBI,
        SUM(r.BB) as total_base_on_balls,
        SUM(r.SO) as total_strike_outs
    FROM (
        SELECT DISTINCT playerID, inducted
        FROM HallofFame
        WHERE inducted = 'Y'
    ) as l
    INNER JOIN (
        SELECT DISTINCT playerID FROM pitching
    ) as m
        ON l.playerID = m.playerID
    INNER JOIN Batting as r
        ON l.playerID = r.playerID
    GROUP BY l.playerID;
    """, con)
display(Markdown(f"### Selected batting totals for pitchers inducted into Hall of Fame"))
display(hof_pitch_bat_stat)


### Selected batting totals for pitchers inducted into Hall of Fame

,playerID,inducted,total_at_bats,total_runs,total_hits,total_home_runs,total_RBI,total_base_on_balls,total_strike_outs
0,alexape01,Y,1810,154,378,11,163,77,276
1,ansonca01,Y,10281,1999,3435,97,2075,984,330
2,becklja01,Y,9551,1603,2938,87,1581,616,526
3,bendech01,Y,1147,102,243,6,116,75,143
4,blylebe01,Y,451,19,59,0,25,5,193
...,...,...,...,...,...,...,...,...,...
103,willivi01,Y,1493,107,248,1,84,81,199
104,wrighge01,Y,2873,665,866,11,326,68,119
105,wrighha01,Y,813,183,224,4,113,37,14
106,wynnea01,Y,1704,136,365,17,173,141,330


### Question 4
Using `pandas` join the previous two tables together by pitcher. (If you want, try to do all of this via SQL! Not required though, feel free to use `pd.merge()` if you'd like) (4 pts)

In [26]:
combined_hof_stats = pd.merge(
    hof_pitchers,
    hof_pitch_bat_stat,
    on = ['playerID','inducted'],
    how = 'outer'
)
display(Markdown(f"### Combined totals for pitching and batting for pitchers inducted into Hall of Fame"))
display(combined_hof_stats)

### Combined totals for pitching and batting for pitchers inducted into Hall of Fame

,playerID,inducted,total_games_started,total_games,total_wins,total_losses,total_outs_pitched,total_complete_games,total_shutouts,total_saves,total_at_bats,total_runs,total_hits,total_home_runs,total_RBI,total_base_on_balls,total_strike_outs
0,alexape01,Y,599,696,373,208,15570,437,90,32,1810,154,378,11,163,77,276
1,ansonca01,Y,0,3,0,1,12,0,0,1,10281,1999,3435,97,2075,984,330
2,becklja01,Y,1,1,0,1,12,0,0,0,9551,1603,2938,87,1581,616,526
3,bendech01,Y,334,459,212,127,9051,255,40,34,1147,102,243,6,116,75,143
4,blylebe01,Y,685,692,287,250,14910,242,60,0,451,19,59,0,25,5,193
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,willivi01,Y,471,513,249,205,11988,388,50,11,1493,107,248,1,84,81,199
104,wrighge01,Y,0,3,0,1,15,0,0,0,2873,665,866,11,326,68,119
105,wrighha01,Y,8,36,4,4,301,0,0,14,813,183,224,4,113,37,14
106,wynnea01,Y,612,691,300,244,13692,290,49,15,1704,136,365,17,173,141,330
